#Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


##Check device availability

## TensorFlow

In [1]:
import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? []


##PyTorch

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.


##TensorFlow - Using GPU Automatically

In [3]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)


Operation completed on: /job:localhost/replica:0/task:0/device:CPU:0


##PyTorch - Manual GPU Transfer

In [4]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)


Tensor 'c' is on device: cpu


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [5]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)




100.0%
100.0%
100.0%
100.0%


##Train on CPU

In [ ]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

##Train on GPU
Change your runtine to T4 GPU and run the following code block

In [ ]:
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()


I suppose we got lesser time for the gpu, it makes more difference on larger models that have more number of layers and filters, gpu speeds up the matrix multiplication due to the presence of numerous small cores.

**So here's an activity for you**
##Use tensorflow to train a model on the MNIST digits dataset on both gpu and cpu and examine which one works faster.

Use your custom number of layers and filters to experiment with the hyperparameters of the model.